In [ ]:
# Extract the European samples from the .psam file

import pandas as pd

# 1. Read the .psam file (using \s+ handles any weird spacing)
df = pd.read_csv(r'D:\Project_CompBio\Summer_2026\QTL_Project\Sex-stratified GWAS\Major Depressive Disorder\LD_Matrices\all_hg38.psam', sep='\s+')

# 2. Filter the dataframe to only keep rows where SuperPop is 'EUR'
eur_df = df[df['SuperPop'] == 'EUR']

# 3. Extract just the ID column
keep_list = eur_df[['#IID']]

# 4. Save this list to a simple text file
keep_list.to_csv('european_samples.txt', index=False, header=False)

print(f"Successfully extracted {len(keep_list)} European samples!")

In [2]:
import pandas as pd
import subprocess
import os

def automate_plink(regions_csv, reference_panel):
    # 1. Read your specific CSV file
    df = pd.read_csv(regions_csv)
    
    # 2. Create a folder to keep your main directory clean
    output_dir = "automated_LD_results_female"
    os.makedirs(output_dir, exist_ok=True)
    
    total_regions = len(df)
    print(f"Starting LD matrix generation for {total_regions} regions...\n")
    
    # 3. Loop through every row in your CSV
    for index, row in df.iterrows():
        # Extract the exact columns you specified
        rs_id = row['rs_id']
        chrom = row['chromosome']
        start_bp = row['from-bp']
        end_bp = row['to-bp']
        
        # Create the exact output name you requested, saved inside the new folder
        out_prefix = os.path.join(output_dir, f"{rs_id}_LD_matrix")
        
        print(f"Processing {index + 1}/{total_regions}: {rs_id} (Chr {chrom})...")
        
        # 4. Construct your exact PLINK2 command
        command = [
            "plink2",
            "--bfile", reference_panel,
            "--chr", str(chrom),
            "--from-bp", str(start_bp),
            "--to-bp", str(end_bp),
            "--maf", "0.01",
            "--r-unphased", "square",
            "--memory", "6000",
            "--out", out_prefix
        ]
        
        # 5. Run the command silently in the background
        try:
            result = subprocess.run(command, capture_output=True, text=True, check=True)
            print("  -> Success!")
            
        except subprocess.CalledProcessError as e:
            # If a region fails (e.g., no SNPs pass the MAF filter), catch it and keep going!
            print(f"  -> ERROR on {rs_id}.")
            if e.stderr:
                print(f"     Details: {e.stderr.strip().splitlines()[-1]}")
            else:
                print("     Details: Unknown error.")

    print(f"\nFinished! All successfully generated matrices are in the '{output_dir}' folder.")

# --- Run the Script ---
# Change 'my_regions.csv' to whatever your CSV file is actually named!
if __name__ == "__main__":
    automate_plink(regions_csv='female_sig_gwas_snps_2mb_window_hg38.csv', reference_panel='1kg_phase3_eur')

Starting LD matrix generation for 16 regions...

Processing 1/16: rs4788616 (Chr 16)...
  -> Success!
Processing 2/16: rs56113727 (Chr 6)...
  -> Success!
Processing 3/16: rs13043844 (Chr 20)...
  -> Success!
Processing 4/16: rs2963222 (Chr 5)...
  -> Success!
Processing 5/16: rs4350429 (Chr 12)...
  -> Success!
Processing 6/16: rs4456268 (Chr 11)...
  -> Success!
Processing 7/16: rs12134194 (Chr 1)...
  -> Success!
Processing 8/16: rs1931263 (Chr 1)...
  -> Success!
Processing 9/16: rs9607805 (Chr 22)...
  -> Success!
Processing 10/16: rs127382 (Chr 1)...
  -> Success!
Processing 11/16: rs11509880 (Chr 7)...
  -> Success!
Processing 12/16: rs10123941 (Chr 9)...
  -> Success!
Processing 13/16: rs1142828 (Chr 5)...
  -> Success!
Processing 14/16: rs1443918 (Chr 13)...
  -> Success!
Processing 15/16: rs10502971 (Chr 18)...
  -> Success!
Processing 16/16: rs11130182 (Chr 3)...
  -> Success!

Finished! All successfully generated matrices are in the 'automated_LD_results_female' folder.


In [1]:
# Code for the sex-combined GWAS

import pandas as pd
import subprocess
import os

def automate_plink(regions_csv, reference_panel):
    # 1. Read your specific CSV file
    df = pd.read_csv(regions_csv)
    
    # 2. NEW: Output folder updated for sex-combined GWAS matrices
    output_dir = "LD_matrices_male_GWAS_common_SNPs"
    os.makedirs(output_dir, exist_ok=True)
    
    total_regions = len(df)
    print(f"Starting LD matrix generation for {total_regions} regions...\n")
    
    # 3. Loop through every row in your CSV
    for index, row in df.iterrows():
        # NEW: Column mapping updated to match 'rsid' exactly
        rsid = row['rsid']
        chrom = row['chromosome']
        start_bp = row['from-bp']
        end_bp = row['to-bp']
        
        # Create the exact output name, saved inside the new folder
        out_prefix = os.path.join(output_dir, f"{rsid}_LD_matrix")
        
        print(f"Processing {index + 1}/{total_regions}: {rsid} (Chr {chrom})...")
        
        # 4. Construct your exact PLINK2 command
        command = [
            "plink2",
            "--bfile", reference_panel,
            "--chr", str(chrom),
            "--from-bp", str(start_bp),
            "--to-bp", str(end_bp),
            "--maf", "0.01",
            "--r-unphased", "square",
            "--memory", "6000",
            "--out", out_prefix
        ]
        
        # 5. Run the command silently in the background
        try:
            result = subprocess.run(command, capture_output=True, text=True, check=True)
            print("  -> Success!")
            
        except subprocess.CalledProcessError as e:
            # Catch errors (e.g., no SNPs pass the MAF filter) and keep the loop alive
            print(f"  -> ERROR on {rsid}.")
            if e.stderr:
                print(f"     Details: {e.stderr.strip().splitlines()[-1]}")
            else:
                print("     Details: Unknown error.")

    print(f"\nFinished! All successfully generated matrices are in the '{output_dir}' folder.")

# --- Run the Script ---
if __name__ == "__main__":
    # Just make sure to change 'snps_with_2mb_windows.csv' to the actual name of your file!
    automate_plink(regions_csv=r'D:\Project_CompBio\Summer_2026\QTL_Project\GWAS\Major Depressive Disorder\Sex-combined GWAS\male_GWAS_common_SNPs_2mb_window_coordinates.csv', reference_panel='1kg_phase3_eur')

Starting LD matrix generation for 2 regions...

Processing 1/2: rs17782683 (Chr 14)...
  -> Success!
Processing 2/2: rs4547421 (Chr 18)...
  -> Success!

Finished! All successfully generated matrices are in the 'LD_matrices_male_GWAS_common_SNPs' folder.
